In [ ]:
import os
import copy as cp
from ase.io import read
import calorine

In [ ]:
# Check which calorine version is being used
print(calorine.__version__)

## Data pre-processing

In [ ]:
training_structures = read('results/bulk/BaTiO3/0082/frozen/G/mode_1/Q_1/structures.xyz@0:')
training_structures[0].get_potential_energy()

In [ ]:
energies = {'Ba': -761.227747, 'Ti': -1604.974503, 'O': -440.177463}

In [ ]:
def shift_energy(structures, energies):

    structures_copy = cp.deepcopy(structures)

    for atoms in structures_copy:
        elements = atoms.get_chemical_symbols()
        atoms.calc.results['energy'] -= sum(energies[element] for element in elements)

    return structures_copy

training_structures = shift_energy(training_structures, energies)


In [ ]:
elements = training_structures[0].get_chemical_symbols()
unique_elements = list(set(elements))
N_elements = len(unique_elements)
print(unique_elements)

In [ ]:
# Join unique elements into one string
unique_elements_str = ' '.join(unique_elements)
print(unique_elements_str)

In [ ]:
from calorine.nep import setup_training

# prepare input for NEP construction
parameters = dict(version=4,
                  type=[N_elements, unique_elements_str],
                  cutoff=[8, 4],
                  neuron=30,
                  generation=100000,
                  batch=1000000)

setup_training(parameters, training_structures,
               rootdir='MLtest', overwrite=True)